# LangChain + OpenAI: Sampling Parameters (`temperature`, `max_tokens`, `top_p`, `top_k`)

Chat models don't just pick the single "best" next token — they **sample** from a probability
distribution over possible next tokens. A handful of parameters control that sampling process and
change how the output looks, even for the exact same prompt.

| Parameter | What it controls |
|---|---|
| `temperature` | How "flat" vs "peaked" the probability distribution is before sampling. Low = deterministic, high = random/creative. |
| `max_tokens` | Hard cap on how many tokens the model is allowed to generate. Output gets cut off once the limit is hit. |
| `top_p` (nucleus sampling) | Only sample from the smallest set of tokens whose cumulative probability is ≥ `top_p`. Lower = fewer candidate tokens. |
| `top_k` | Only sample from the `k` most likely next tokens. **Not supported by OpenAI's API** — see the note in that section. |

This notebook uses `langchain_openai.ChatOpenAI` with model `gpt-5.4-nano` throughout.

## Setup

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

PROMPT = "Write a one-sentence tagline for a coffee shop."

## 1. `temperature`

Range is typically `0.0`–`2.0` for OpenAI models.

- **Low (`0`)**: near-deterministic, picks the highest-probability token almost every time. Good for factual/consistent answers.
- **High (`1.5`–`2.0`)**: flattens the distribution so lower-probability tokens get picked more often. More varied, more "creative", but can get incoherent at the extreme.

To see the effect, we ask the **same prompt 3 times** at each temperature. Low temperature should give near-identical outputs; high temperature should give visibly different ones.

In [3]:
for temp in [0.0, 1.0, 1.9]:
    print(f"=== temperature={temp} ===")
    llm = ChatOpenAI(model="gpt-5.4-nano", temperature=temp)
    for i in range(2):
        response = llm.invoke(PROMPT)
        print(f"{i+1}. {response.content}")
    print()

=== temperature=0.0 ===
1. Brewed with care, poured with joy—your next favorite cup starts here.
2. Sip slow, brew bold—your new favorite cup awaits.

=== temperature=1.0 ===
1. Sip slowly, savor boldly—fresh coffee, warm vibes, every day.
2. Sip slow, taste bold—your neighborhood coffee stop.

=== temperature=1.9 ===
1. Sip slow, taste local, and let every cup feel like home.
2. Sip handcrafted coffee in every cup—warm, bold, and brewed just for you.



## 2. `max_tokens`

Caps the number of tokens the model is allowed to generate in its response. Once the cap is hit,
generation stops mid-output — the response is **not** summarized to fit, it's just truncated.

Check `response.response_metadata["finish_reason"]`: it will say `"length"` when the model was
cut off by `max_tokens`, versus `"stop"` when it finished naturally.

In [3]:
llm = ChatOpenAI(model="gpt-5.4-nano")

for max_tokens in [16, 50, 300]:
    response = llm.invoke(
        "Explain how a bicycle works.",
        max_tokens=max_tokens,
    )
    finish_reason = response.response_metadata.get("finish_reason")
    print(f"=== max_tokens={max_tokens} (finish_reason={finish_reason}) ===")
    print(response.content)
    print()

=== max_tokens=16 (finish_reason=length) ===
A bicycle is basically a system of **simple machines** working together



=== max_tokens=50 (finish_reason=length) ===
A bicycle is basically a machine for turning human muscle power into forward motion efficiently. It works by combining a few key systems: gears, wheels, and brakes.

## 1) You pedal → you create rotational force
When you press



=== max_tokens=300 (finish_reason=length) ===
A bicycle works by turning a force from you (pedaling) into motion, using gears, wheels, and braking.

### 1) You apply power with your legs
- When you pedal, your legs apply force to the **crank** (the part the pedals attach to).
- That turns the **front chainring** (usually a sprocket) connected to the crank.

### 2) The chain transfers that turning to the rear wheel
- A **chain** connects the front chainring to the **rear sprocket** (cog).
- As the chain moves, it turns the rear sprocket, which turns the **rear wheel**.
- The rear wheel’s rotation is what moves the bike forward.

### 3) Gears control how hard or easy it is to pedal
- **Gear ratio** is determined by which front and rear sprockets you use.
  - **Large front / small rear** (higher gear): easier to go fast, but harder to pedal.
  - **Small front / large rear** (lower gear): easier to pedal, but you go slower.
- Gearing changes how many times the pedals turn for each turn of 

## 3. `top_p` (nucleus sampling)

Instead of considering *every* possible next token, `top_p` restricts sampling to the smallest
group of tokens whose combined probability reaches `top_p`. Range is `0.0`–`1.0`.

- `top_p=1.0`: consider (almost) all tokens — no restriction.
- `top_p=0.1`: only the most probable tokens, whose probabilities sum to 10%, are eligible — very narrow, near-deterministic.

`temperature` and `top_p` both affect randomness but in different ways — OpenAI recommends altering
one or the other, not both at once. Here we fix `temperature=1.0` and vary only `top_p`.

In [4]:
for top_p in [0.1, 0.5, 1.0]:
    print(f"=== top_p={top_p} ===")
    llm = ChatOpenAI(model="gpt-5.4-nano", temperature=1.0, top_p=top_p)
    for i in range(3):
        response = llm.invoke(PROMPT)
        print(f"{i+1}. {response.content}")
    print()

=== top_p=0.1 ===
1. Sip slow, savor bold—your perfect cup starts here.
2. Sip slow, savor bold—your perfect cup starts here.
3. Sip slowly, savor boldly—your perfect cup starts here.

=== top_p=0.5 ===
1. Sip slowly, savor locally—your perfect cup starts here.
2. Sip slow, savor bold—your perfect cup starts here.
3. Sip slowly, savor boldly—your perfect cup starts here.

=== top_p=1.0 ===
1. Savor every sip at our cozy coffee shop—freshly brewed, warmly served, and always minutes from your next favorite cup.
2. Brewed fresh, poured with heart—your perfect cup is waiting.
3. Sip slow, savor bold—your new favorite cup awaits.



## 4. `top_k`

`top_k` restricts sampling to only the `k` most probable next tokens (e.g. `top_k=5` → only the
top 5 candidate tokens are eligible, everything else gets zero probability). It's supported by
some providers LangChain wraps (e.g. Anthropic, HuggingFace) — but **OpenAI's Chat Completions
API has no `top_k` parameter**. `ChatOpenAI` has no `top_k` field for this reason; OpenAI only
exposes `temperature` and `top_p` for controlling randomness.

To prove this rather than just assert it: `ChatOpenAI` accepts a `model_kwargs` dict that gets
forwarded as extra keyword arguments to the underlying `openai` Python client's
`chat.completions.create()` call. If we sneak `top_k` in through there, the OpenAI SDK itself
rejects it before a request is even sent — `top_k` isn't a parameter it (or the API behind it)
recognizes.

In [8]:
llm = ChatOpenAI(model="gpt-5.4-nano", model_kwargs={"top_k": 5})

try:
    llm.invoke(PROMPT)
except Exception as e:
    print(f"{type(e).__name__}: {e}")

TypeError: Completions.create() got an unexpected keyword argument 'top_k'. Did you mean 'top_p'?
